In [1]:
from typing import Any

from kirin.ir import Method
import matplotlib.pyplot as plt
import numpy as np

from bloqade import squin, tsim
from bloqade.cirq_utils import emit_circuit, load_circuit, noise
from bloqade.pyqrack import StackMemorySimulator
from bloqade.types import MeasurementResult, Qubit
from kirin.dialects.ilist import IList
import matplotlib.pyplot as plt

# this will help us have return types for our methods that have more intuitive names
Register = IList[Qubit, Any]
Measurement = IList[MeasurementResult, Any]

# this function will help us visualize some circuits
def show_circuit(squin_kernel):
    @squin.kernel
    def _to_visualize():
        _ = squin_kernel()

    return tsim.Circuit(_to_visualize).diagram(height=400)

In [2]:
@squin.kernel
def test_t_gate() -> Register:
    qubits = squin.qalloc(1)
    
    # 1. Ponemos el qubit en superposición
    squin.h(qubits[0])
    
    # 2. Aplicamos la puerta T (prueba primero squin.t(qubits[0]) si existe)
    # Si falla, usamos su equivalente Rz(pi/4)
    squin.t(qubits[0])
    #squin.rz(np.pi/4, qubits[0]) 
    
    return qubits
# Configuramos el simulador de PyQrack (min_qubits=1 para un solo qubit)
sim = StackMemorySimulator(min_qubits=1)
task = sim.task(test_t_gate)

# Ejecutamos y extraemos la matriz de densidad
result_state = task.run()
rho = StackMemorySimulator.reduced_density_matrix(result_state)

print("Matriz de densidad tras aplicar H y T (o Rz):")
print(rho)

Matriz de densidad tras aplicar H y T (o Rz):
[[0.5       +0.j         0.35355342-0.35355339j]
 [0.35355342+0.35355339j 0.50000004+0.j        ]]


In [5]:
@squin.kernel
def circuito_dos_qubits() -> Register:
    # Asignamos 2 qubits
    qubits = squin.qalloc(2)
    
    # Aplicamos una puerta Hadamard al primer qubit (índice 0)
    squin.h(qubits[0])
    
    # Aplicamos una puerta CNOT controlada por el qubit 0 hacia el qubit 1
    squin.cx(qubits[0], qubits[1])
    
    # Retornamos los qubits sin medir para no colapsar el estado cuántico
    return qubits

# 3. Simulación y extracción del estado (Matriz de Densidad)
simulador = StackMemorySimulator(min_qubits=2)
tarea = simulador.task(circuito_dos_qubits)
estado_resultante = tarea.run()

matriz_densidad = StackMemorySimulator.reduced_density_matrix(estado_resultante)

print("Matriz de densidad del sistema de 2 qubits:")
# Redondeamos un poco para que sea más legible en la terminal
print(np.round(matriz_densidad, 3))

# 4. Visualización del circuito
dibujo = show_circuit(circuito_dos_qubits)

# Si estás en Jupyter Notebook, ejecutar 'dibujo' en la última línea de una celda 
# renderizará el SVG directamente.
dibujo

Matriz de densidad del sistema de 2 qubits:
[[0.5+0.j 0. +0.j 0. +0.j 0.5+0.j]
 [0. +0.j 0. +0.j 0. +0.j 0. +0.j]
 [0. +0.j 0. +0.j 0. +0.j 0. +0.j]
 [0.5+0.j 0. +0.j 0. +0.j 0.5+0.j]]


In [6]:
@squin.kernel
def circuito_completo_medida() -> Measurement:
    qubits = squin.qalloc(2)
    
    # Preparamos un estado entrelazado con H y CNOT
    squin.h(qubits[0])
    squin.cx(qubits[0], qubits[1])
    
    # Aplicamos la puerta S al qubit 0 (completando el set Clifford+T)
    squin.s(qubits[0])
    #squin.rz(np.pi/2, qubits[0]) 
    
    # Medimos todos los qubits del registro
    bits_clasicos = squin.broadcast.measure(qubits)
    
    return bits_clasicos

# Configuración del simulador
simulador = StackMemorySimulator(min_qubits=2)
tarea = simulador.task(circuito_completo_medida)

# 1. Ejecutar un solo "shot" (intento)
un_intento = tarea.run()
print("Resultado de 1 solo intento:", un_intento)

# 2. Ejecutar estadísticas (ej: 1000 shots)
resultados_estadisticos = tarea.batch_run(shots=1000)
print("Resultados de 1000 intentos:", resultados_estadisticos)

Resultado de 1 solo intento: IList([<Measurement.One: 1>, <Measurement.One: 1>])
Resultados de 1000 intentos: {(<Measurement.One: 1>, <Measurement.One: 1>): 0.489, (<Measurement.Zero: 0>, <Measurement.Zero: 0>): 0.511}
